# mfd_volinflowrate_m 因子

某只股票在某个交易日内，主力资金在“成交量维度”上的净流入强度

## 市值行业中性化后因子指标计算

In [ ]:
# -*- coding: utf-8 -*-
"""
BigQuant 复现华泰资金流向因子 mfd_volinflowrate_m：主力净流入率（量）（市值行业中性化版本）。

口径：
1. mfd_volinflowrate_m ≈ cn_stock_moneyflow.netflow_volume_rate_main。
2. 主力净流入率（量）为正向因子，原始因子值直接取 netflow_volume_rate_main，不取反。
3. 每个截面对原始因子做 MAD 去极值、标准化，再对 log(流通市值) 和行业哑变量做中性化，最后对中性化残差再次标准化。
4. 以 10 个交易日作为截面周期，使用中性化后的因子计算未来 10 个交易日收益对应的 IC、RankIC、回归因子收益率和 t 值。
5. 股票池剔除 ST、当前停牌、下一交易日停牌；使用 cn_stock_factors_base 的后复权 close 计算收益。
6. 回归法参考研报：未来 10 日相对沪深300超额收益 ~ 行业哑变量 + 中性化后因子，WLS 权重为 sqrt(流通市值)。

优化要点：
1. 在 DAI SQL 端完成交易日抽样、未来收益、停牌过滤和基准收益计算，只把调仓截面传回 Python。
2. Python 端只保留必要列，并将 instrument/industry 转为 category，数值列转为 float32，降低内存占用。
3. 截面中性化和截面回归全部使用 NumPy 矩阵运算，避免逐截面反复创建 statsmodels 对象。
4. 不输出日志，只 display 汇总表，并画 IC 与 RankIC 同图。
"""

import os
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager
from IPython.display import display

try:
    import dai
except ImportError:
    from bigquant import dai  # type: ignore

warnings.filterwarnings("ignore")


@dataclass(frozen=True)
class Config:
    start_date: str = "2020-01-01"       # cn_stock_moneyflow 官网数据起点为 2015-01-01
    end_date: str = "2026-06-30"
    forward_days: int = 10
    rebalance_freq: int = 10
    min_cross_section_size: int = 30
    industry_col: str = "sw2021_level1"  # 可改 sw2014_level1；取决于账号内 cn_stock_factors_base 字段
    factor_name: str = "mfd_volinflowrate_m_neutral"
    factor_source_col: str = "netflow_volume_rate_main"
    chinese_font_path: str = ""          # 若 BigQuant 环境仍无法显示中文，可上传中文字体文件后填入绝对路径


CFG = Config()


def _font_has_chinese(font_path: str) -> bool:
    """粗略判断字体是否包含常用中文字形，避免误选英文字体。"""
    try:
        ft = font_manager.get_font(font_path)
        cmap = ft.get_charmap()
        return all(ord(ch) in cmap for ch in "因子日期相关系数")
    except Exception:
        return False


def _candidate_font_paths(user_font_path: str = "") -> List[str]:
    """优先使用用户指定字体，其次在 BigQuant/Jupyter/Linux/Windows/macOS 常见路径中搜索中文字体。"""
    candidates: List[str] = []

    if user_font_path:
        candidates.append(user_font_path)

    env_font = os.environ.get("CHINESE_FONT_PATH", "")
    if env_font:
        candidates.append(env_font)

    direct_paths = [
        "./SimHei.ttf",
        "./simhei.ttf",
        "./msyh.ttc",
        "./Microsoft YaHei.ttf",
        "./NotoSansCJK-Regular.ttc",
        "./NotoSansCJKsc-Regular.otf",
        "/home/jovyan/work/SimHei.ttf",
        "/home/jovyan/work/NotoSansCJK-Regular.ttc",
        "/usr/share/fonts/truetype/wqy/wqy-microhei.ttc",
        "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc",
        "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.otf",
        "/usr/share/fonts/opentype/noto/NotoSansCJKsc-Regular.otf",
        "/usr/share/fonts/truetype/noto/NotoSansCJK-Regular.ttc",
        "/usr/share/fonts/truetype/arphic/uming.ttc",
        "/System/Library/Fonts/PingFang.ttc",
        "C:/Windows/Fonts/msyh.ttc",
        "C:/Windows/Fonts/simhei.ttf",
    ]
    candidates.extend(direct_paths)

    search_roots = [
        Path.cwd(),
        Path.home(),
        Path("/usr/share/fonts"),
        Path("/usr/local/share/fonts"),
        Path("/opt/conda/lib/python3.10/site-packages/matplotlib/mpl-data/fonts/ttf"),
    ]
    name_keywords = (
        "NotoSansCJK",
        "NotoSansSC",
        "SourceHanSans",
        "SourceHanSerif",
        "WenQuanYi",
        "wqy",
        "SimHei",
        "simhei",
        "msyh",
        "PingFang",
        "Arial Unicode",
    )
    suffixes = {".ttf", ".ttc", ".otf"}

    for root in search_roots:
        if not root.exists():
            continue
        try:
            for p in root.rglob("*"):
                if p.suffix.lower() in suffixes and any(k.lower() in p.name.lower() for k in name_keywords):
                    candidates.append(str(p))
        except Exception:
            continue

    seen = set()
    unique = []
    for p in candidates:
        pp = str(Path(p).expanduser())
        if pp not in seen:
            seen.add(pp)
            unique.append(pp)
    return unique


def set_chinese_font(font_path: str = "") -> Optional[font_manager.FontProperties]:
    """设置 Matplotlib 中文字体，并返回可显式传入标题/坐标轴/图例的 FontProperties。"""
    preferred_names = [
        "Microsoft YaHei",
        "SimHei",
        "Noto Sans CJK SC",
        "Noto Sans SC",
        "Source Han Sans SC",
        "WenQuanYi Micro Hei",
        "PingFang SC",
        "Arial Unicode MS",
    ]

    plt.rcParams["axes.unicode_minus"] = False
    plt.rcParams["pdf.fonttype"] = 42
    plt.rcParams["ps.fonttype"] = 42
    plt.rcParams["svg.fonttype"] = "none"

    for path in _candidate_font_paths(font_path):
        if not Path(path).exists():
            continue
        if not _font_has_chinese(path):
            continue
        try:
            font_manager.fontManager.addfont(path)
            prop = font_manager.FontProperties(fname=path)
            font_name = prop.get_name()
            plt.rcParams["font.family"] = "sans-serif"
            plt.rcParams["font.sans-serif"] = [font_name] + preferred_names + ["DejaVu Sans"]
            return prop
        except Exception:
            continue

    available = {f.name for f in font_manager.fontManager.ttflist}
    for name in preferred_names:
        if name in available:
            plt.rcParams["font.family"] = "sans-serif"
            plt.rcParams["font.sans-serif"] = [name] + [n for n in preferred_names if n != name] + ["DejaVu Sans"]
            return font_manager.FontProperties(family=name)

    plt.rcParams["font.family"] = "sans-serif"
    plt.rcParams["font.sans-serif"] = preferred_names + ["DejaVu Sans"]
    warnings.warn(
        "当前运行环境未找到可用中文字体。若图表仍显示方框，请上传 SimHei.ttf、msyh.ttc "
        "或 NotoSansCJK-Regular.ttc，并在 CFG.chinese_font_path 中填入该字体路径。",
        RuntimeWarning,
    )
    return None


def fetch_signal_panel(cfg: Config) -> pd.DataFrame:
    """只读取调仓截面所需数据，减少传输量和内存占用。"""
    fetch_end = (pd.Timestamp(cfg.end_date) + pd.Timedelta(days=max(90, cfg.forward_days * 12))).strftime("%Y-%m-%d")

    sql = f"""
    PRAGMA enable_pushdown_window;

    WITH trading_dates AS (
        SELECT
            date,
            ROW_NUMBER() OVER (ORDER BY date) AS rn
        FROM (
            SELECT DISTINCT date
            FROM cn_stock_factors_base
            WHERE date >= DATE '{cfg.start_date}'
              AND date <= DATE '{cfg.end_date}'
        )
    ),

    signal_dates AS (
        SELECT date
        FROM trading_dates
        WHERE MOD(rn - 1, {cfg.rebalance_freq}) = 0
    ),

    base AS (
        SELECT
            date,
            instrument,
            close,
            float_market_cap,
            {cfg.industry_col} AS industry_level1,
            st_status,
            suspended,
            list_sector,
            LEAD(close, {cfg.forward_days}) OVER (PARTITION BY instrument ORDER BY date) AS close_fwd,
            LEAD(suspended, 1) OVER (PARTITION BY instrument ORDER BY date) AS next_suspended
        FROM cn_stock_factors_base
        WHERE date >= DATE '{cfg.start_date}'
          AND date <= DATE '{fetch_end}'
          AND list_sector != 4
    ),

    bench_raw AS (
        SELECT
            date,
            MAX(hs300_close) AS hs300_close
        FROM cn_stock_factors_base
        WHERE date >= DATE '{cfg.start_date}'
          AND date <= DATE '{fetch_end}'
        GROUP BY date
    ),

    bench AS (
        SELECT
            date,
            hs300_close,
            LEAD(hs300_close, {cfg.forward_days}) OVER (ORDER BY date) AS hs300_close_fwd
        FROM bench_raw
    )

    SELECT
        b.date,
        b.instrument,
        b.float_market_cap,
        b.industry_level1,
        mf.{cfg.factor_source_col} AS factor_raw,
        b.close_fwd / b.close - 1.0 AS ret_fwd_10d,
        b.close_fwd / b.close - 1.0 - (be.hs300_close_fwd / be.hs300_close - 1.0) AS excess_ret_fwd_10d
    FROM base AS b
    JOIN signal_dates AS sd
      ON b.date = sd.date
    JOIN cn_stock_moneyflow AS mf
      ON b.date = mf.date AND b.instrument = mf.instrument
    JOIN bench AS be
      ON b.date = be.date
    WHERE b.date <= DATE '{cfg.end_date}'
      AND b.st_status = 0
      AND b.suspended = 0
      AND COALESCE(b.next_suspended, 1) = 0
      AND b.close > 0
      AND b.close_fwd > 0
      AND be.hs300_close > 0
      AND be.hs300_close_fwd > 0
      AND b.float_market_cap > 0
      AND mf.{cfg.factor_source_col} IS NOT NULL
    ORDER BY b.date, b.instrument
    """

    df = dai.query(sql, filters={"date": [cfg.start_date, fetch_end]}).df()
    if df.empty:
        raise ValueError("查询结果为空，请检查日期区间、权限或字段名。")

    df["date"] = pd.to_datetime(df["date"]).dt.normalize()
    df["instrument"] = df["instrument"].astype("category")
    df["industry_level1"] = df["industry_level1"].fillna("未知").astype("category")

    for col in ["float_market_cap", "factor_raw", "ret_fwd_10d", "excess_ret_fwd_10d"]:
        df[col] = pd.to_numeric(df[col], errors="coerce", downcast="float")

    df = df.dropna(subset=["float_market_cap", "factor_raw", "ret_fwd_10d", "excess_ret_fwd_10d"])
    return df.reset_index(drop=True)


def robust_zscore_np(x: np.ndarray) -> np.ndarray:
    """MAD 去极值后标准化。"""
    x = x.astype(np.float64, copy=False)
    out = np.full(x.shape, np.nan, dtype=np.float64)
    valid = np.isfinite(x)
    if valid.sum() < 3:
        return out

    xv = x[valid]
    med = np.nanmedian(xv)
    mad = np.nanmedian(np.abs(xv - med))
    if np.isfinite(mad) and mad > 1e-12:
        scale = 1.4826 * mad
        lo, hi = med - 3.0 * scale, med + 3.0 * scale
    else:
        lo, hi = np.nanpercentile(xv, [1.0, 99.0])

    xv = np.clip(xv, lo, hi)
    std = xv.std(ddof=0)
    if np.isfinite(std) and std > 1e-12:
        out[valid] = (xv - xv.mean()) / std
    return out


def _neutralize_array_by_cap_industry(
    y: np.ndarray,
    float_market_cap: np.ndarray,
    industry: pd.Series,
) -> np.ndarray:
    """y 对 log(流通市值) 和行业哑变量做截面中性化，返回残差。"""
    y = y.astype(np.float64, copy=False)
    log_cap = np.log(float_market_cap.astype(np.float64, copy=False).clip(min=1.0))
    dummies = industry_dummies(industry)
    X = np.column_stack([np.ones(len(y), dtype=np.float64), log_cap, dummies])

    valid = np.isfinite(y) & np.isfinite(X).all(axis=1)
    resid = np.full(len(y), np.nan, dtype=np.float64)
    if valid.sum() < max(10, X.shape[1] + 2):
        return resid

    Xv = X[valid]
    yv = y[valid]
    beta = np.linalg.lstsq(Xv, yv, rcond=None)[0]
    resid[valid] = yv - Xv @ beta
    return resid


def add_neutralized_factor(df: pd.DataFrame) -> pd.DataFrame:
    """
    构造最终测试因子：
    1. 原始主力净流入率（量）因子不取反：factor_raw = netflow_volume_rate_main；
    2. 每个调仓截面对 factor_raw 做 MAD 去极值和标准化；
    3. 将标准化后的因子对 log(流通市值) 和行业哑变量做中性化；
    4. 对中性化残差再次做 MAD 去极值和标准化，得到 factor_neutral_z。
    """
    df = df.copy()
    n = len(df)
    factor_z = np.full(n, np.nan, dtype=np.float32)
    factor_neutral = np.full(n, np.nan, dtype=np.float32)
    factor_neutral_z = np.full(n, np.nan, dtype=np.float32)

    raw_values = df["factor_raw"].to_numpy(dtype=np.float64, copy=False)
    cap_values = df["float_market_cap"].to_numpy(dtype=np.float64, copy=False)

    for _, idx in df.groupby("date", sort=False, observed=True).indices.items():
        idx_arr = np.asarray(idx)

        z = robust_zscore_np(raw_values[idx_arr])
        factor_z[idx_arr] = z.astype(np.float32)

        resid = _neutralize_array_by_cap_industry(
            y=z,
            float_market_cap=cap_values[idx_arr],
            industry=df["industry_level1"].iloc[idx_arr],
        )
        factor_neutral[idx_arr] = resid.astype(np.float32)

        # 中性化残差再标准化，保证最终因子在每个截面上可比较。
        neutral_z = robust_zscore_np(resid)
        factor_neutral_z[idx_arr] = neutral_z.astype(np.float32)

    df["factor_z_before_neutral"] = factor_z
    df["factor_neutral"] = factor_neutral
    df["factor_neutral_z"] = factor_neutral_z
    df = df.dropna(subset=["factor_neutral_z"]).reset_index(drop=True)
    return df


def corr_np(x: np.ndarray, y: np.ndarray) -> float:
    valid = np.isfinite(x) & np.isfinite(y)
    if valid.sum() < 3:
        return np.nan
    xv = x[valid].astype(np.float64, copy=False)
    yv = y[valid].astype(np.float64, copy=False)
    xv = xv - xv.mean()
    yv = yv - yv.mean()
    denom = np.sqrt(np.dot(xv, xv) * np.dot(yv, yv))
    if not np.isfinite(denom) or denom <= 1e-18:
        return np.nan
    return float(np.dot(xv, yv) / denom)


def rank_np(x: np.ndarray) -> np.ndarray:
    return pd.Series(x).rank(method="average").to_numpy(dtype=np.float64, copy=False)


def industry_dummies(industry: pd.Series) -> np.ndarray:
    """行业哑变量，drop_first=True；返回 float64 矩阵。"""
    codes = pd.Categorical(industry).codes
    n = len(codes)
    k = int(codes.max()) + 1
    if k <= 1:
        return np.empty((n, 0), dtype=np.float64)

    mat = np.zeros((n, k - 1), dtype=np.float64)
    rows = np.arange(n)
    mask = codes > 0
    mat[rows[mask], codes[mask] - 1] = 1.0
    return mat


def wls_factor_return(g: pd.DataFrame) -> Tuple[float, float]:
    """未来 10 日超额收益 ~ 中性化后因子 + 行业哑变量；WLS 权重为 sqrt(流通市值)。"""
    y = g["excess_ret_fwd_10d"].to_numpy(dtype=np.float64, copy=False)
    f = g["factor_neutral_z"].to_numpy(dtype=np.float64, copy=False)
    dummies = industry_dummies(g["industry_level1"])
    X = np.column_stack([np.ones(len(g), dtype=np.float64), f, dummies])
    w = np.sqrt(g["float_market_cap"].to_numpy(dtype=np.float64, copy=False).clip(min=1.0))

    valid = np.isfinite(y) & np.isfinite(X).all(axis=1) & np.isfinite(w) & (w > 0)
    if valid.sum() < max(30, X.shape[1] + 5):
        return np.nan, np.nan

    Xv = X[valid]
    yv = y[valid]
    wv = w[valid]

    xtwx = Xv.T @ (wv[:, None] * Xv)
    xtwy = Xv.T @ (wv * yv)
    xtwx_inv = np.linalg.pinv(xtwx, rcond=1e-12)
    beta = xtwx_inv @ xtwy

    resid = yv - Xv @ beta
    rank = np.linalg.matrix_rank(xtwx)
    dof = max(len(yv) - rank, 1)
    sigma2 = float(np.sum(wv * resid * resid) / dof)
    se = np.sqrt(np.maximum(np.diag(sigma2 * xtwx_inv), 0.0))

    factor_ret = float(beta[1])
    t_value = float(beta[1] / se[1]) if se[1] > 1e-18 else np.nan
    return factor_ret, t_value


def calc_cross_section_metrics(g: pd.DataFrame, min_n: int) -> Optional[Dict[str, float]]:
    if len(g) < min_n:
        return None

    factor = g["factor_neutral_z"].to_numpy(dtype=np.float64, copy=False)
    ret = g["ret_fwd_10d"].to_numpy(dtype=np.float64, copy=False)
    valid = np.isfinite(factor) & np.isfinite(ret)
    if valid.sum() < min_n:
        return None

    ic = corr_np(factor[valid], ret[valid])
    rank_ic = corr_np(rank_np(factor[valid]), rank_np(ret[valid]))
    factor_ret, t_value = wls_factor_return(g)

    return {
        "date": g["date"].iloc[0],
        "IC": ic,
        "RankIC": rank_ic,
        "因子收益率": factor_ret,
        "t值": t_value,
    }


def calc_all_metrics(data: pd.DataFrame, cfg: Config) -> pd.DataFrame:
    rows: List[Dict[str, float]] = []
    for _, g in data.groupby("date", sort=True, observed=True):
        row = calc_cross_section_metrics(g, cfg.min_cross_section_size)
        if row is not None:
            rows.append(row)
    if not rows:
        raise ValueError("没有足够截面可计算指标，请检查区间、股票池或最小截面样本数。")
    return pd.DataFrame(rows).sort_values("date").reset_index(drop=True)


def safe_ir(s: pd.Series) -> float:
    s = pd.to_numeric(s, errors="coerce").dropna()
    std = s.std(ddof=1)
    if len(s) < 2 or not np.isfinite(std) or std <= 1e-18:
        return np.nan
    return float(s.mean() / std)


def make_summary(metrics: pd.DataFrame, cfg: Config) -> pd.DataFrame:
    summary = pd.DataFrame([{
        "因子": cfg.factor_name,
        "起始日": metrics["date"].min().strftime("%Y-%m-%d"),
        "结束日": metrics["date"].max().strftime("%Y-%m-%d"),
        "截面数": int(metrics["date"].nunique()),
        "IC均值": metrics["IC"].mean(),
        "ICIR": safe_ir(metrics["IC"]),
        "RankIC均值": metrics["RankIC"].mean(),
        "RankICIR": safe_ir(metrics["RankIC"]),
        "因子收益率": metrics["因子收益率"].mean(),
        "t值": metrics["t值"].mean(),
    }])
    return summary


def format_summary(summary: pd.DataFrame) -> pd.DataFrame:
    out = summary.copy()
    decimal_cols = ["IC均值", "ICIR", "RankIC均值", "RankICIR", "因子收益率", "t值"]
    for col in decimal_cols:
        out[col] = out[col].map(lambda x: "" if pd.isna(x) else f"{x:.6f}")
    return out


def plot_ic_rankic(metrics: pd.DataFrame, cfg: Config) -> None:
    font_prop = set_chinese_font(cfg.chinese_font_path)
    fig, ax = plt.subplots(figsize=(14, 6))
    ax.plot(metrics["date"], metrics["IC"], label="IC", linewidth=1.6)
    ax.plot(metrics["date"], metrics["RankIC"], label="RankIC", linewidth=1.6)
    ax.axhline(0, linewidth=1.0, linestyle="--")

    title = f"{cfg.factor_name}：IC 与 RankIC 时序图"
    if font_prop is not None:
        ax.set_title(title, fontproperties=font_prop)
        ax.set_xlabel("日期", fontproperties=font_prop)
        ax.set_ylabel("相关系数", fontproperties=font_prop)
        ax.legend(prop=font_prop)
    else:
        ax.set_title(title)
        ax.set_xlabel("日期")
        ax.set_ylabel("相关系数")
        ax.legend()

    ax.grid(True, alpha=0.3)
    fig.autofmt_xdate()
    plt.tight_layout()
    plt.show()


def main() -> Tuple[pd.DataFrame, pd.DataFrame]:
    data = fetch_signal_panel(CFG)
    data = add_neutralized_factor(data)
    metrics = calc_all_metrics(data, CFG)
    summary = make_summary(metrics, CFG)

    display(format_summary(summary))
    plot_ic_rankic(metrics, CFG)
    return summary, metrics


summary, metrics = main()


该因子即使从相关指标上来看也十分不适合作为选股排序的因子，因此不再做更多的探究